# OpenMontage — Colab GPU Run

This notebook installs runtime dependencies and runs a GPU-aware OpenMontage pipeline on Google Colab.
It will prompt for the GitHub repository URL (your repo) so you can clone and run the code.

Notes:
- Use a GPU runtime: Runtime -> Change runtime type -> GPU.
- Artifacts are saved under `projects/<project>/` inside the Colab VM; download them to your local machine or to Drive if desired.


In [ ]:
# 1) Install system packages (FFmpeg, libsndfile)
# These are required for audio/video handling.
!apt-get update -qq
!apt-get install -y -qq ffmpeg libsndfile1 git-lfs
!git lfs install --local || true


In [ ]:
# 2) Clone the repository
REPO_URL = input('Enter your GitHub repo URL (https://github.com/<user>/<repo>.git): ').strip()
if not REPO_URL:
    raise SystemExit('No repository URL provided. Upload the notebook into your repo or provide the URL to clone.')
import os
if not os.path.exists('/content/openmontage'):
    !git clone {REPO_URL} /content/openmontage
%cd /content/openmontage
!git status --porcelain || true


In [ ]:
# 3) Install Python packages (PyTorch + ML/audio/video libs)
# Upgrade pip first
!python -m pip install -q --upgrade pip
# Install CUDA-enabled PyTorch for Colab (adjust index if runtime changes).
!python -m pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
# Install core ML/media libraries used by the project
!python -m pip install -q faster-whisper diffusers transformers accelerate safetensors ffmpeg-python moviepy librosa scipy huggingface_hub pyyaml python-dotenv pydantic requests
# Optional: speedups (uncomment at your discretion)
# !python -m pip install -q xformers bitsandbytes


In [ ]:
# 4) Verify GPU availability and torch
import torch
print('Torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device name:', torch.cuda.get_device_name(0))


In [ ]:
# 5) Run preflight health check and a pipeline run
# This will run the preflight adapter health check and then run an OpenMontage pipeline.
# It may download model weights depending on what adapters are enabled in configs/models.yaml.
from pathlib import Path
import json, time
import sys
sys.path.insert(0, '/content/openmontage')
from scripts.model_cache import health_check
from scripts.colab_run import main as colab_main
# Run health check
hc = health_check()
print('Health check results:', hc)
# Run the colab runner (it will run the pipeline and optionally benchmark).
# Use the notebook command-line invocation: set args via the environment if needed.
# For convenience, call the runner directly from Python with default args.
from importlib import reload
import scripts.colab_run as cr
reload(cr)
# If you want benchmarking, re-run this cell as: !python scripts/colab_run.py --project colab-run --benchmark
print('Running pipeline (mock=False) via scripts/colab_run.py...')
!python scripts/colab_run.py --project colab-colabrun --benchmark
print('Done. Artifacts in projects/colab-colabrun/')


## Next steps
- Download `projects/colab-colabrun/renders/final.mp4` from the Colab Files explorer.
- Benchmarks will be under `outputs/benchmarks/`.
- If an adapter falls back to mock, check `configs/models.yaml` and install the model or SDK you prefer.
